### Импорты и загрузка

In [1]:
import sys
sys.path.append('../src')

import torch
import pandas as pd
import numpy as np
from pathlib import Path
import json

from dataset import build_item_vocab, build_user_sequences
from model import create_model
from metrics import build_eval_targets, build_context, evaluate_model, recall_at_k, ndcg_at_k, average_precision_at_k
from my_evaluate import GPT2Predictor, evaluate_gpt2_fast

PROC = Path('../data/processed')
RESULTS = Path('../results')

### Загрузкка чекпоинта

In [2]:
train = pd.read_parquet(PROC / 'train.parquet')
val = pd.read_parquet(PROC / 'val.parquet')
test = pd.read_parquet(PROC / 'test.parquet')

item2idx, idx2item = build_item_vocab(train)
vocab_size = len(item2idx) + 2

# Рождение модели и загрузка веса
model = create_model(vocab_size=vocab_size)
state = torch.load(RESULTS / 'checkpoints' / 'best_model.pt', map_location='cpu')
model.load_state_dict(state)
print("Модель загружена")

# Просмотр весов (проверка)
n_params = sum(p.numel() for p in model.parameters())
print(f"Параметров: {n_params:,}")

[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 14936), got 50256. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 14936), got 50256. This may result in unexpected behavior.


Модель загружена
Параметров: 2,315,136


### Таргеты и контексты: подготовка

In [3]:
val_targets = build_eval_targets(val)
test_targets = build_eval_targets(test)

val_context = build_context(train)
test_context = build_context(train, val)

print(f"Val targets:  {len(val_targets):,}")
print(f"Test targets: {len(test_targets):,}")

Val targets:  176,224
Test targets: 186,690


### Быстрый тест (1000 юзеров)

In [4]:
# Предиктор родился
predictor = GPT2Predictor(model, item2idx, idx2item, max_len=50)

# Набор из 1000 юзеров из val_targets
val_small = val_targets[val_targets['user_id'].isin(
    val_targets['user_id'].unique()[:1000]
)]
print(f"Юзеров в тесте: {val_small['user_id'].nunique()}")

# Inference
preds_dict = evaluate_gpt2_fast(predictor, val_small, val_context, k=100)

# Рассчёт метрик
def compute_metrics(preds_dict, targets_df, k_values=(10, 50, 100)):
    max_k = max(k_values)
    accum = {f'{m}@{k}': [] for k in k_values for m in ['Recall', 'NDCG', 'MAP']}
    for _, row in targets_df.iterrows():
        preds = preds_dict.get(row['user_id'], [])[:max_k]
        target = row['item_id']
        for k in k_values:
            accum[f'Recall@{k}'].append(recall_at_k(preds, target, k))
            accum[f'NDCG@{k}'].append(ndcg_at_k(preds, target, k))
            accum[f'MAP@{k}'].append(average_precision_at_k(preds, target, k))
    return {k: float(np.mean(v)) for k, v in accum.items()}

m = compute_metrics(preds_dict, val_small)
print("\n=== GPT-2 на 1000 юзерах (val) ===")
for k, v in m.items():
    print(f"{k:>12s}: {v:.4f}")

Юзеров в тесте: 1000
Уникальных юзеров для оценки: 1,000


predict_batch:   0%|          | 0/4 [00:00<?, ?it/s]


=== GPT-2 на 1000 юзерах (val) ===
   Recall@10: 0.1658
     NDCG@10: 0.0945
      MAP@10: 0.0726
   Recall@50: 0.3197
     NDCG@50: 0.1279
      MAP@50: 0.0795
  Recall@100: 0.4101
    NDCG@100: 0.1426
     MAP@100: 0.0808


### Полноценная оценка для всего test

In [5]:
# Inference на всех юзерах test
print("Стартует inference на test...")
test_preds = evaluate_gpt2_fast(predictor, test_targets, test_context, k=100, batch_size=512)
print(f"Готово: {len(test_preds)} предсказаний")

# Расчёт метрик
test_metrics = compute_metrics(test_preds, test_targets)
print("\n=== GPT-2 на test ===")
for k, v in test_metrics.items():
    print(f"{k:>12s}: {v:.4f}")

Стартует inference на test...
Уникальных юзеров для оценки: 132,323


predict_batch:   0%|          | 0/259 [00:00<?, ?it/s]

Готово: 132323 предсказаний

=== GPT-2 на test ===
   Recall@10: 0.2016
     NDCG@10: 0.1198
      MAP@10: 0.0946
   Recall@50: 0.3564
     NDCG@50: 0.1536
      MAP@50: 0.1017
  Recall@100: 0.4457
    NDCG@100: 0.1681
     MAP@100: 0.1029


### Сравнение с baseline-ами

In [6]:
# Загрузка результатов бейзлайнов
baselines = pd.read_csv(RESULTS / 'baselines_v2_metrics.csv', index_col=0)
gpt2_series = pd.Series(test_metrics, name='GPT-2')

# Добавление в таблицу
combined = baselines.copy()
combined['GPT-2'] = gpt2_series

print("=== Сводная таблица ===")
print(combined.round(4).to_string())

combined.to_csv(RESULTS / 'gpt2_vs_baselines.csv')

# Краткий вывод
pop_ndcg10 = baselines.loc['NDCG@10', 'Popularity']
knn_ndcg10 = baselines.loc['NDCG@10', 'Item-kNN']
gpt2_ndcg10 = test_metrics['NDCG@10']

print(f"\nNDCG@10:")
print(f"  Popularity: {pop_ndcg10:.4f}")
print(f"  Item-kNN:   {knn_ndcg10:.4f}")
print(f"  GPT-2:      {gpt2_ndcg10:.4f}")
print(f"  Прирост к Popularity: {(gpt2_ndcg10/pop_ndcg10 - 1)*100:+.1f}%")
print(f"  Прирост к Item-kNN:   {(gpt2_ndcg10/knn_ndcg10 - 1)*100:+.1f}%")

=== Сводная таблица ===
            Popularity  Recent-Popularity  Last-Item+FB  Pure Last-Item  Item-kNN   GPT-2
metric                                                                                   
Recall@10       0.1643             0.1601        0.1614             0.0    0.1842  0.2016
NDCG@10         0.0901             0.0846        0.0720             0.0    0.1091  0.1198
MAP@10          0.0670             0.0614        0.0443             0.0    0.0859  0.0946
Recall@50       0.2807             0.2622        0.2807             0.0    0.3402  0.3564
NDCG@50         0.1150             0.1066        0.0975             0.0    0.1429  0.1536
MAP@50          0.0719             0.0659        0.0493             0.0    0.0929  0.1017
Recall@100      0.3490             0.3325        0.3482             0.0    0.4298  0.4457
NDCG@100        0.1261             0.1180        0.1084             0.0    0.1575  0.1681
MAP@100         0.0729             0.0669        0.0503             0.0    0